In [1]:
# 1. Simplified self-attention

# Step 1.0 - Load input word embeddings (each row = one word vector)
import torch

inputs = torch.tensor([
    [0.43, 0.15, 0.89],  # "your"
    [0.55, 0.87, 0.66],  # "journey"
    [0.57, 0.85, 0.64],  # "starts" ← our query word for next step
    [0.22, 0.58, 0.33],  # "with"
    [0.77, 0.25, 0.10],  # "one"
    [0.05, 0.80, 0.55]   # "step"
])

In [6]:
# 1.1 Computing Attention Weights for inputs[2] (word: "starts")

# 1.1.1 Attention Scores (dot product)
# - Using inputs[2] ("starts") as query vector
query = inputs[2]

# Computing attention scores (similarity between query and all other word vectors)
attn_scores_2 = torch.zeros(len(inputs))  # initialize a score array

for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)  # dot product between input and query

# Printing the raw attention scores
print("Attention Scores for inputs[2]:", attn_scores_2)


Attention Scores for inputs[2]: tensor([0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605])


In [7]:
# 1.1.2 Attention Weights (applying Softmax)

# Converting scores to probabilities using softmax
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)

# Printing attention weights (how much attention each word gets)
print("Attention Weights for inputs[2]:", attn_weights_2)

Attention Weights for inputs[2]: tensor([0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565])


In [8]:
# 1.1.3 Context Vector (weighted sum of input vectors)

# Computing context vector (weighted average of input vectors)
context_vector_2 = torch.zeros(3)  # initializes a 3D vector (same size as word vectors)

for i in range(len(inputs)):
    context_vector_2 += attn_weights_2[i] * inputs[i]  # weighted sum

# Printing final context vector (new refined meaning for word "starts")
print("Context Vector for inputs[2]:", context_vector_2)


Context Vector for inputs[2]: tensor([0.4431, 0.6496, 0.5671])


In [10]:
# 1.2 Computing Attention Weights for All Inputs
# Computing attention scores for every word against every other word (attention score Matrix)

# Each row in this matrix will contain the dot products of one word (as query) with all words (as keys)
attn_scores_matrix = torch.zeros((len(inputs), len(inputs)))  # shape: (num_words, num_words)

# Loop over each word as query
for i in range(len(inputs)):
    query = inputs[i]
    for j in range(len(inputs)):
        attn_scores_matrix[i][j] = torch.dot(query, inputs[j])

# Printing the full attention scores matrix
print("Attention Scores Matrix (query vs keys):")
print(attn_scores_matrix)

Attention Scores Matrix (query vs keys):
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [12]:
# 1.2.2 Attention Weights Matrix (Softmax per row)
#Converting scores to attention weights using softmax across each row

# This gives us a probability distribution for each query over all keys
attn_weights_matrix = torch.softmax(attn_scores_matrix, dim=1)

# Print the attention weights matrix
print("Attention Weights Matrix (after softmax):")
print(attn_weights_matrix)

Attention Weights Matrix (after softmax):
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [14]:
# 1.2.3 All Context Vectors (weighted sum per row)
#Computing a context vector for each word using its attention weights

# Each context vector is the weighted sum of all input vectors
context_vectors = torch.zeros_like(inputs)  # same shape as original inputs

for i in range(len(inputs)):
    for j in range(len(inputs)):
        context_vectors[i] += attn_weights_matrix[i][j] * inputs[j]

# Printing all context vectors
print("Context Vectors for All Words:")
print(context_vectors)


Context Vectors for All Words:
tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [ ]:
#  2. The ‘Self’ in Self-Attention

2.1 – Weight Parameters vs Attention Weights
1. Weight Parameters (Wq, Wk, Wv):
These are the learnable matrices in a self-attention mechanism.
They are used to transform the input word vectors into:

- Queries (Q) → What a word is looking for.
- Keys (K) → How a word describes itself.
- Values (V) → What content the word carries.

2. Attention Weights (we did this in question 1):
These are not learned, they are computed dynamically for each input.
They come from:

- Calculating dot products between Q and K (similarity).
- Applying softmax to get probabilities.
- They tell us how much attention each word gives to every other word.

Key difference:

- Weight Parameters = fixed numbers that are learned during training.
- Attention Weights = dynamic numbers that change for every input sentence.

- Example:
- If I say “The cat sat on the mat”, the attention weights will be different for each word, depending on context.
-But the weight matrices (Wq, Wk, Wv) stay the same — they are the model’s internal settings that get updated only during training.


In [15]:
# 2.2 Computing Weight Parameters for Inputs[2]:

# Step 2.2.1 - Initialize learnable weight matrices Wq, Wk, Wv
# Each one transforms input vectors into queries, keys, and values

# assuming we want to project 3D input vectors to 3D outputs (same size for simplicity)
Wq = torch.tensor([
    [0.2, 0.1, 0.4],
    [0.3, 0.5, 0.2],
    [0.6, 0.1, 0.3]
])  # Weight matrix for queries

Wk = torch.tensor([
    [0.3, 0.4, 0.2],
    [0.1, 0.6, 0.3],
    [0.5, 0.2, 0.4]
])  # Weight matrix for keys

Wv = torch.tensor([
    [0.2, 0.3, 0.6],
    [0.7, 0.5, 0.1],
    [0.4, 0.2, 0.3]
])  # Weight matrix for values

In [16]:
# 2.2.2 – Compute query, key, and value vectors for inputs[1] (the word "journey")
# Selecting the input vector for the word "journey"

# This is inputs[1] = [0.55, 0.87, 0.66]
word_vector = inputs[1]

# Compute query vector: Q = Wq × input
query_vector = torch.matmul(Wq, word_vector)

# Compute key vector: K = Wk × input
key_vector = torch.matmul(Wk, word_vector)

# Compute value vector: V = Wv × input
value_vector = torch.matmul(Wv, word_vector)

# Print the transformed vectors
print("Query vector (Q) for 'journey':", query_vector)
print("Key vector (K) for 'journey':", key_vector)
print("Value vector (V) for 'journey':", value_vector)

Query vector (Q) for 'journey': tensor([0.4610, 0.7320, 0.6150])
Key vector (K) for 'journey': tensor([0.6450, 0.7750, 0.7130])
Value vector (V) for 'journey': tensor([0.7670, 0.8860, 0.5920])


In [20]:
# 2.2.3 Compute the Attention Score inputs[1][1] (ω₁₁)
# Computing the attention score between the query and key of "journey" (ω₁₁)

# Attention score is a dot product between query and key vectors
attention_score_11 = torch.dot(query_vector, key_vector)

# Printing the score
print("Attention Score ω₁₁ (journey attends to itself):", attention_score_11)

Attention Score ω₁₁ (journey attends to itself): tensor(1.3031)


In [22]:
# 2.2.4 Compute all Attention Scores for inputs[1] (journey → all keys)
# Computing attention scores between "journey" (as query) and all other words (as keys)

# Initialize an array to store the scores
attention_scores_for_journey = torch.zeros(len(inputs))

# Loop through all inputs to compute key vectors for each word
for i in range(len(inputs)):
    key_i = torch.matmul(Wk, inputs[i])            # key vector for word i
    attention_scores_for_journey[i] = torch.dot(query_vector, key_i)  # dot product with journey's query

# Printing the attention scores: how much "journey" attends to each word
print("Attention scores (journey's query vs all keys):")
print(attention_scores_for_journey)

Attention scores (journey's query vs all keys):
tensor([0.8316, 1.3031, 1.2874, 0.7313, 0.6421, 0.9300])


In [23]:
# 2.2.5 Attention Weights for inputs[1] (Softmax)
#Converting the raw attention scores into attention weights using softmax

attention_weights_for_journey = torch.softmax(attention_scores_for_journey, dim=0)

# Printing the final attention weights (how much attention "journey" gives to each word)
print("Attention Weights (journey attends to all words):")
print(attention_weights_for_journey)

Attention Weights (journey attends to all words):
tensor([0.1425, 0.2284, 0.2249, 0.1289, 0.1179, 0.1573])


In [24]:
# 2.2.6 Calculate Context Vector for inputs[1]
#  Computing value vectors for all inputs (using Wv)
# Then calculating the final context vector for "journey" using attention weights

context_vector_for_journey = torch.zeros(3)  # same dimension as input vectors

for i in range(len(inputs)):
    value_i = torch.matmul(Wv, inputs[i])  # compute value vector for word i
    context_vector_for_journey += attention_weights_for_journey[i] * value_i  # weighted sum

# Print the final context vector (the refined representation of "journey")
print("Context Vector for 'journey':")
print(context_vector_for_journey)


Context Vector for 'journey':
tensor([0.6183, 0.6864, 0.4738])


In [ ]:
# 2.3 Computing Weight Parameters for All Inputs
# 2.3.1 was already covered when we defined Wq, Wk, Wv earlier

In [25]:
# 2.3.2 Compute Query, Key, and Value vectors for all words
# Computing Q, K, V vectors for all input words using Wq, Wk, Wv

# Initializing new tensors to store the results
queries = torch.zeros_like(inputs)
keys = torch.zeros_like(inputs)
values = torch.zeros_like(inputs)

# Looping over each word to compute Q, K, V
for i in range(len(inputs)):
    queries[i] = torch.matmul(Wq, inputs[i])  # query vector
    keys[i] = torch.matmul(Wk, inputs[i])     # key vector
    values[i] = torch.matmul(Wv, inputs[i])   # value vector

# Printing the query, key, and value matrices
print("All Query Vectors (Q):")
print(queries)

print("All Key Vectors (K):")
print(keys)

print("All Value Vectors (V):")
print(values)

All Query Vectors (Q):
tensor([[0.4570, 0.3820, 0.5400],
        [0.4610, 0.7320, 0.6150],
        [0.4550, 0.7240, 0.6190],
        [0.2340, 0.4220, 0.2890],
        [0.2190, 0.3760, 0.5170],
        [0.3100, 0.5250, 0.2750]])
All Key Vectors (K):
tensor([[0.3670, 0.4000, 0.6010],
        [0.6450, 0.7750, 0.7130],
        [0.6390, 0.7590, 0.7110],
        [0.3640, 0.4690, 0.3580],
        [0.3510, 0.2570, 0.4750],
        [0.4450, 0.6500, 0.4050]])
All Value Vectors (V):
tensor([[0.6650, 0.4650, 0.4690],
        [0.7670, 0.8860, 0.5920],
        [0.7530, 0.8880, 0.5900],
        [0.4160, 0.4770, 0.3030],
        [0.2890, 0.6740, 0.3880],
        [0.5800, 0.4900, 0.3450]])


In [26]:
# 2.3.3 Compute Attention Scores for All Inputs (Q × Kᵗ)
# Computing full attention score matrix by dotting Q with Kᵗ

# Initializing matrix: attention_scores[i][j] = dot(Q_i, K_j)
attention_scores = torch.zeros((len(inputs), len(inputs)))

# Computing dot product between each Q_i and each K_j
for i in range(len(inputs)):
    for j in range(len(inputs)):
        attention_scores[i][j] = torch.dot(queries[i], keys[j])

# Printing the full attention score matrix
print("Attention Scores Matrix (Q × Kᵗ):")
print(attention_scores)

Attention Scores Matrix (Q × Kᵗ):
tensor([[0.6451, 0.9758, 0.9659, 0.5388, 0.5151, 0.6704],
        [0.8316, 1.3031, 1.2874, 0.7313, 0.6421, 0.9300],
        [0.8286, 1.2959, 1.2804, 0.7268, 0.6398, 0.9238],
        [0.4284, 0.6840, 0.6753, 0.3866, 0.3279, 0.4955],
        [0.5415, 0.8013, 0.7929, 0.4411, 0.4191, 0.5512],
        [0.4890, 0.8029, 0.7921, 0.4575, 0.3744, 0.5906]])


In [27]:
# 2.3.4 Attention Weights for All Inputs (Softmax)
# Converting attention scores into attention weights using softmax

# Applying softmax on each row (dim=1) to get probabilities
attention_weights = torch.softmax(attention_scores, dim=1)

# Printing the final attention weights matrix
print("Attention Weights Matrix (after softmax):")
print(attention_weights)

Attention Weights Matrix (after softmax):
tensor([[0.1521, 0.2118, 0.2097, 0.1368, 0.1336, 0.1560],
        [0.1425, 0.2284, 0.2249, 0.1289, 0.1179, 0.1573],
        [0.1429, 0.2280, 0.2245, 0.1291, 0.1183, 0.1572],
        [0.1537, 0.1985, 0.1968, 0.1474, 0.1390, 0.1644],
        [0.1567, 0.2032, 0.2015, 0.1417, 0.1386, 0.1582],
        [0.1495, 0.2046, 0.2024, 0.1448, 0.1333, 0.1654]])


In [28]:
# 2.3.5 Calculate Context Vectors for All Inputs
# Computing final context vectors (self-attention output) for all words

# Initializing a matrix to store the context vectors (same shape as inputs)
context_vectors = torch.zeros_like(inputs)

# For each word (row), we compute the weighted sum of all value vectors using attention weights
for i in range(len(inputs)):
    for j in range(len(inputs)):
        context_vectors[i] += attention_weights[i][j] * values[j]

# Printing final context vectors for all words
print("Final Context Vectors (Self-Attention Output):")
print(context_vectors)

Final Context Vectors (Self-Attention Output):
tensor([[0.6075, 0.6763, 0.4675],
        [0.6183, 0.6864, 0.4738],
        [0.6180, 0.6862, 0.4737],
        [0.5996, 0.6668, 0.4611],
        [0.6026, 0.6704, 0.4640],
        [0.6034, 0.6705, 0.4633]])
